In [1]:
%pip install streamlit langchain langchain-google-genai google-generativeai beautifulsoup4 reportlab python-dotenv plotly tldextract

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import os
import pandas as pd
import tldextract
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import google.generativeai as gena

C:\Users\ARPAN\AppData\Local\Temp\ipykernel_15744\3311333098.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as gena


In [3]:
sample_email = """
From: support@amaz0n-security.xyz
To: arpan@gmail.com
Subject: Verify your account immediately

Dear Customer,

Your account has been suspended.

Please verify your account immediately.

Click here:
http://amaz0n-login-security.xyz

Failure to verify within 24 hours will permanently suspend your account.

Regards,
Amazon Security Team
"""

In [4]:
print(sample_email)


From: support@amaz0n-security.xyz
To: arpan@gmail.com
Subject: Verify your account immediately

Dear Customer,

Your account has been suspended.

Please verify your account immediately.

Click here:
http://amaz0n-login-security.xyz

Failure to verify within 24 hours will permanently suspend your account.

Regards,
Amazon Security Team



In [5]:
sender = re.search(r"From:\s*(.*)", sample_email)

if sender:
    sender = sender.group(1)
else:
    sender = ""

print(sender)

support@amaz0n-security.xyz


In [6]:
receiver = re.search(r"To:\s*(.*)", sample_email)

if receiver:
    receiver = receiver.group(1)
else:
    receiver = ""

print(receiver)

arpan@gmail.com


In [7]:
subject = re.search(r"Subject:\s*(.*)", sample_email)

if subject:
    subject = subject.group(1)
else:
    subject = ""

print(subject)

Verify your account immediately


In [8]:
urls = re.findall(r"https?://[^\s]+", sample_email)

print(urls)

['http://amaz0n-login-security.xyz']


In [9]:
body = sample_email.split("\n\n", 1)[1]

print(body)

Dear Customer,

Your account has been suspended.

Please verify your account immediately.

Click here:
http://amaz0n-login-security.xyz

Failure to verify within 24 hours will permanently suspend your account.

Regards,
Amazon Security Team



In [10]:
email_data = {
    "sender": sender,
    "receiver": receiver,
    "subject": subject,
    "body": body,
    "urls": urls
}

email_data

{'sender': 'support@amaz0n-security.xyz',
 'receiver': 'arpan@gmail.com',
 'subject': 'Verify your account immediately',
 'body': 'Dear Customer,\n\nYour account has been suspended.\n\nPlease verify your account immediately.\n\nClick here:\nhttp://amaz0n-login-security.xyz\n\nFailure to verify within 24 hours will permanently suspend your account.\n\nRegards,\nAmazon Security Team\n',
 'urls': ['http://amaz0n-login-security.xyz']}

In [11]:
df = pd.DataFrame([email_data])

df

,sender,receiver,subject,body,urls
0,support@amaz0n-security.xyz,arpan@gmail.com,Verify your account immediately,"Dear Customer,\n\nYour account has been suspen...",[http://amaz0n-login-security.xyz]


In [12]:
# Suspicious Top Level Domains (TLDs)
suspicious_tlds = [
    "xyz",
    "top",
    "click",
    "online",
    "shop",
    "site",
    "info"
]

# Common brand impersonations
fake_brands = {
    "amaz0n": "amazon",
    "paypa1": "paypal",
    "micr0soft": "microsoft",
    "g00gle": "google",
    "faceb00k": "facebook",
    "app1e": "apple"
}

# Free email providers
free_email_domains = [
    "gmail.com",
    "yahoo.com",
    "hotmail.com",
    "outlook.com"
]

In [13]:
def analyze_sender(sender):

    result = {
        "risk_score": 0,
        "risk_level": "Safe",
        "findings": []
    }

    sender = sender.lower()

    # Split username and domain
    if "@" not in sender:
        result["risk_score"] += 40
        result["findings"].append("Invalid email format.")
        return result

    username, domain = sender.split("@")

    # -----------------------------
    # Check 1: Suspicious TLD
    # -----------------------------
    tld = domain.split(".")[-1]

    if tld in suspicious_tlds:
        result["risk_score"] += 30
        result["findings"].append(
            f"Suspicious domain extension: .{tld}"
        )

    # -----------------------------
    # Check 2: Fake Brand
    # -----------------------------
    for fake, real in fake_brands.items():

        if fake in domain:

            result["risk_score"] += 40

            result["findings"].append(
                f"Possible impersonation of {real}"
            )

    # -----------------------------
    # Check 3: Company using Gmail
    # -----------------------------
    company_names = [
        "amazon",
        "paypal",
        "google",
        "microsoft",
        "apple"
    ]

    for company in company_names:

        if company in username:

            if domain in free_email_domains:

                result["risk_score"] += 35

                result["findings"].append(
                    f"{company.title()} is using a free email service."
                )

    # -----------------------------
    # Decide Risk Level
    # -----------------------------
    score = result["risk_score"]

    if score >= 70:
        result["risk_level"] = "High"

    elif score >= 40:
        result["risk_level"] = "Medium"

    else:
        result["risk_level"] = "Low"

    return result

In [14]:
analyze_sender(sender)

{'risk_score': 70,
 'risk_level': 'High',
 'findings': ['Suspicious domain extension: .xyz',
  'Possible impersonation of amazon']}

In [15]:
analyze_sender("amazonhelp@gmail.com")

{'risk_score': 35,
 'risk_level': 'Low',
 'findings': ['Amazon is using a free email service.']}

In [16]:
analyze_sender("security@paypal.com")

{'risk_score': 0, 'risk_level': 'Low', 'findings': []}

In [17]:
analyze_sender("support@paypa1-login.xyz")

{'risk_score': 70,
 'risk_level': 'High',
 'findings': ['Suspicious domain extension: .xyz',
  'Possible impersonation of paypal']}

In [18]:
# Suspicious TLDs
suspicious_tlds = [
    "xyz", "top", "click", "site", "online", "info"
]

# URL shorteners
shorteners = [
    "bit.ly",
    "tinyurl.com",
    "goo.gl",
    "t.co",
    "ow.ly",
    "is.gd",
    "rb.gy"
]

# Fake brand spellings
brand_keywords = {
    "amaz0n": "Amazon",
    "paypa1": "PayPal",
    "micr0soft": "Microsoft",
    "g00gle": "Google",
    "app1e": "Apple"
}

In [19]:
import re

def analyze_url(url):

    result = {
        "url": url,
        "risk_score": 0,
        "risk_level": "Safe",
        "findings": []
    }

    url = url.lower()

    # -----------------------------
    # HTTP instead of HTTPS
    # -----------------------------
    if url.startswith("http://"):
        result["risk_score"] += 20
        result["findings"].append("Uses HTTP instead of HTTPS.")

    # -----------------------------
    # IP Address URL
    # -----------------------------
    if re.search(r"https?://\d+\.\d+\.\d+\.\d+", url):
        result["risk_score"] += 35
        result["findings"].append("URL uses an IP address.")

    # -----------------------------
    # Suspicious TLD
    # -----------------------------
    domain = url.split("//")[-1].split("/")[0]

    if "." in domain:
        tld = domain.split(".")[-1]

        if tld in suspicious_tlds:
            result["risk_score"] += 25
            result["findings"].append(f"Suspicious TLD: .{tld}")

    # -----------------------------
    # URL Shortener
    # -----------------------------
    for short in shorteners:
        if short in domain:
            result["risk_score"] += 20
            result["findings"].append("Shortened URL detected.")

    # -----------------------------
    # Fake Brand
    # -----------------------------
    for fake, real in brand_keywords.items():
        if fake in url:
            result["risk_score"] += 35
            result["findings"].append(f"Possible {real} impersonation.")

    # -----------------------------
    # Very Long URL
    # -----------------------------
    if len(url) > 100:
        result["risk_score"] += 10
        result["findings"].append("Very long URL.")

    # -----------------------------
    # Multiple Hyphens
    # -----------------------------
    if domain.count("-") >= 2:
        result["risk_score"] += 10
        result["findings"].append("Too many hyphens in domain.")

    # -----------------------------
    # Decide Risk Level
    # -----------------------------
    score = result["risk_score"]

    if score >= 70:
        result["risk_level"] = "High"
    elif score >= 40:
        result["risk_level"] = "Medium"
    else:
        result["risk_level"] = "Low"

    return result

In [20]:
analyze_url("http://amaz0n-login-security.xyz")

{'url': 'http://amaz0n-login-security.xyz',
 'risk_score': 90,
 'risk_level': 'High',
 'findings': ['Uses HTTP instead of HTTPS.',
  'Suspicious TLD: .xyz',
  'Possible Amazon impersonation.',
  'Too many hyphens in domain.']}

In [21]:
analyze_url("https://www.google.com")

{'url': 'https://www.google.com',
 'risk_score': 0,
 'risk_level': 'Low',
 'findings': []}

In [22]:
analyze_url("http://192.168.1.10/login")

{'url': 'http://192.168.1.10/login',
 'risk_score': 55,
 'risk_level': 'Medium',
 'findings': ['Uses HTTP instead of HTTPS.', 'URL uses an IP address.']}

In [23]:
analyze_url("https://bit.ly/abc123")

{'url': 'https://bit.ly/abc123',
 'risk_score': 20,
 'risk_level': 'Low',
 'findings': ['Shortened URL detected.']}

In [24]:
analyze_url("https://paypa1-security.xyz/login")

{'url': 'https://paypa1-security.xyz/login',
 'risk_score': 60,
 'risk_level': 'Medium',
 'findings': ['Suspicious TLD: .xyz', 'Possible PayPal impersonation.']}

In [25]:
url_results = []

for url in email_data["urls"]:
    url_results.append(analyze_url(url))

url_results

[{'url': 'http://amaz0n-login-security.xyz',
  'risk_score': 90,
  'risk_level': 'High',
  'findings': ['Uses HTTP instead of HTTPS.',
   'Suspicious TLD: .xyz',
   'Possible Amazon impersonation.',
   'Too many hyphens in domain.']}]

In [26]:
url_results = []

for url in email_data["urls"]:
    url_results.append(analyze_url(url))

url_results

[{'url': 'http://amaz0n-login-security.xyz',
  'risk_score': 90,
  'risk_level': 'High',
  'findings': ['Uses HTTP instead of HTTPS.',
   'Suspicious TLD: .xyz',
   'Possible Amazon impersonation.',
   'Too many hyphens in domain.']}]

In [27]:
# Urgent words
urgent_words = [
    "urgent", "immediately", "within 24 hours",
    "act now", "verify now", "limited time",
    "expires today", "as soon as possible"
]

# Credential request words
credential_words = [
    "password", "otp", "pin", "login",
    "username", "account verification",
    "security code", "cvv"
]

# Threat words
threat_words = [
    "account suspended",
    "account blocked",
    "legal action",
    "your account will be closed",
    "unauthorized access",
    "failure to verify"
]

# Financial words
financial_words = [
    "bank",
    "payment",
    "credit card",
    "debit card",
    "transaction",
    "refund",
    "invoice"
]

# Reward scam words
reward_words = [
    "winner",
    "lottery",
    "prize",
    "gift",
    "claim reward",
    "congratulation"
]

In [28]:
def analyze_content(body):

    result = {
        "risk_score": 0,
        "risk_level": "Safe",
        "findings": []
    }

    text = body.lower()

    # -------------------------
    # Urgent language
    # -------------------------
    for word in urgent_words:
        if word in text:
            result["risk_score"] += 10
            result["findings"].append(f"Urgent language: '{word}'")

    # -------------------------
    # Credential requests
    # -------------------------
    for word in credential_words:
        if word in text:
            result["risk_score"] += 15
            result["findings"].append(f"Credential request: '{word}'")

    # -------------------------
    # Threats
    # -------------------------
    for word in threat_words:
        if word in text:
            result["risk_score"] += 15
            result["findings"].append(f"Threat detected: '{word}'")

    # -------------------------
    # Financial keywords
    # -------------------------
    for word in financial_words:
        if word in text:
            result["risk_score"] += 10
            result["findings"].append(f"Financial keyword: '{word}'")

    # -------------------------
    # Lottery / Reward
    # -------------------------
    for word in reward_words:
        if word in text:
            result["risk_score"] += 10
            result["findings"].append(f"Reward scam keyword: '{word}'")

    # -------------------------
    # Excessive Exclamation Marks
    # -------------------------
    if body.count("!") >= 3:
        result["risk_score"] += 5
        result["findings"].append("Too many exclamation marks.")

    # -------------------------
    # ALL CAPS Detection
    # -------------------------
    words = body.split()

    capital_words = [
        w for w in words
        if len(w) > 3 and w.isupper()
    ]

    if len(capital_words) >= 3:
        result["risk_score"] += 5
        result["findings"].append("Too many capitalized words.")

    # -------------------------
    # Risk Level
    # -------------------------
    score = result["risk_score"]

    if score >= 70:
        result["risk_level"] = "High"

    elif score >= 40:
        result["risk_level"] = "Medium"

    else:
        result["risk_level"] = "Low"

    return result

In [29]:
analyze_content(email_data["body"])

{'risk_score': 50,
 'risk_level': 'Medium',
 'findings': ["Urgent language: 'immediately'",
  "Urgent language: 'within 24 hours'",
  "Credential request: 'login'",
  "Threat detected: 'failure to verify'"]}

In [30]:
safe_email = """
Hello Arpan,

Your monthly statement is available.

Thank you for using our services.

Regards,
ABC Bank
"""

analyze_content(safe_email)

{'risk_score': 10,
 'risk_level': 'Low',
 'findings': ["Financial keyword: 'bank'"]}

In [31]:
phishing = """
URGENT!!

Your account has been suspended.

Verify your password immediately!!

Failure to verify within 24 hours will permanently suspend your account.

Click here now!!
"""

analyze_content(phishing)

{'risk_score': 65,
 'risk_level': 'Medium',
 'findings': ["Urgent language: 'urgent'",
  "Urgent language: 'immediately'",
  "Urgent language: 'within 24 hours'",
  "Credential request: 'password'",
  "Threat detected: 'failure to verify'",
  'Too many exclamation marks.']}

In [32]:
dangerous_extensions = [
    ".exe",
    ".scr",
    ".bat",
    ".cmd",
    ".js",
    ".vbs",
    ".jar",
    ".msi"
]

macro_extensions = [
    ".docm",
    ".xlsm",
    ".pptm"
]

compressed_extensions = [
    ".zip",
    ".rar",
    ".7z"
]

In [33]:
def analyze_attachments(attachments):

    result = {
        "risk_score": 0,
        "risk_level": "Safe",
        "findings": []
    }

    # No attachment
    if len(attachments) == 0:
        result["findings"].append("No attachments found.")
        return result

    for file in attachments:

        file = file.lower()

        # Dangerous executable
        for ext in dangerous_extensions:

            if file.endswith(ext):
                result["risk_score"] += 40
                result["findings"].append(
                    f"Dangerous executable: {file}"
                )

        # Macro-enabled Office file
        for ext in macro_extensions:

            if file.endswith(ext):
                result["risk_score"] += 25
                result["findings"].append(
                    f"Macro-enabled Office file: {file}"
                )

        # Compressed archive
        for ext in compressed_extensions:

            if file.endswith(ext):
                result["risk_score"] += 15
                result["findings"].append(
                    f"Compressed archive: {file}"
                )

        # Double extension
        if ".pdf.exe" in file or ".doc.exe" in file:

            result["risk_score"] += 40
            result["findings"].append(
                f"Double extension detected: {file}"
            )

    score = result["risk_score"]

    if score >= 70:
        result["risk_level"] = "High"

    elif score >= 40:
        result["risk_level"] = "Medium"

    else:
        result["risk_level"] = "Low"

    return result

In [34]:
attachments = []

analyze_attachments(attachments)

{'risk_score': 0, 'risk_level': 'Safe', 'findings': ['No attachments found.']}

In [35]:
attachments = [
    "invoice.pdf"
]

analyze_attachments(attachments)

{'risk_score': 0, 'risk_level': 'Low', 'findings': []}

In [36]:
attachments = [
    "invoice.pdf"
]

analyze_attachments(attachments)

{'risk_score': 0, 'risk_level': 'Low', 'findings': []}

In [37]:
attachments = [
    "invoice.pdf.exe"
]

analyze_attachments(attachments)

{'risk_score': 80,
 'risk_level': 'High',
 'findings': ['Dangerous executable: invoice.pdf.exe',
  'Double extension detected: invoice.pdf.exe']}

In [38]:
attachments = [
    "salary.xlsm"
]

analyze_attachments(attachments)

{'risk_score': 25,
 'risk_level': 'Low',
 'findings': ['Macro-enabled Office file: salary.xlsm']}

In [39]:
attachments = [
    "documents.zip"
]

analyze_attachments(attachments)

{'risk_score': 15,
 'risk_level': 'Low',
 'findings': ['Compressed archive: documents.zip']}

In [40]:
def calculate_final_risk(sender_result,
                         url_results,
                         content_result,
                         attachment_result):

    total_score = 0
    findings = []

    # Sender
    total_score += sender_result["risk_score"]
    findings.extend(sender_result["findings"])

    # URLs
    for result in url_results:
        total_score += result["risk_score"]
        findings.extend(result["findings"])

    # Content
    total_score += content_result["risk_score"]
    findings.extend(content_result["findings"])

    # Attachments
    total_score += attachment_result["risk_score"]
    findings.extend(attachment_result["findings"])

    # Cap the score at 100
    total_score = min(total_score, 100)

    if total_score >= 80:
        verdict = "🚨 PHISHING"

    elif total_score >= 60:
        verdict = "⚠ HIGH RISK"

    elif total_score >= 30:
        verdict = "🟡 SUSPICIOUS"

    else:
        verdict = "🟢 SAFE"

    return {
        "final_score": total_score,
        "verdict": verdict,
        "findings": findings
    }

In [41]:
sender_result = analyze_sender(email_data["sender"])

url_results = [
    analyze_url(url)
    for url in email_data["urls"]
]

content_result = analyze_content(email_data["body"])

# Sample attachments (replace later with parsed attachments)
attachments = ["invoice.pdf.exe"]

attachment_result = analyze_attachments(attachments)

final_result = calculate_final_risk(
    sender_result,
    url_results,
    content_result,
    attachment_result
)

final_result

{'final_score': 100,
 'verdict': '🚨 PHISHING',
 'findings': ['Suspicious domain extension: .xyz',
  'Possible impersonation of amazon',
  'Uses HTTP instead of HTTPS.',
  'Suspicious TLD: .xyz',
  'Possible Amazon impersonation.',
  'Too many hyphens in domain.',
  "Urgent language: 'immediately'",
  "Urgent language: 'within 24 hours'",
  "Credential request: 'login'",
  "Threat detected: 'failure to verify'",
  'Dangerous executable: invoice.pdf.exe',
  'Double extension detected: invoice.pdf.exe']}

In [42]:
%pip install langchain langchain-google-genai google-generativeai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [43]:
%pip install -U langchain langchain-core langchain-community langchain-groq groq python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Using cached groq-1.5.0-py3-none-any.whl.metadata (16 kB)
Note: you may need to restart the kernel to use updated packages.


In [44]:
#  I am using GROQ because it has tree tier , realiability and tool calling
import os

os.environ["GROQ_API_KEY"] = "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"

In [45]:
%pip install -U langchain langchain-core langchain-community langchain-groq groq python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Using cached groq-1.5.0-py3-none-any.whl.metadata (16 kB)
Note: you may need to restart the kernel to use updated packages.


In [46]:
print(os.environ["GROQ_API_KEY"])

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


In [47]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [48]:
response = llm.invoke("What is phishing?")

print(response.content)

Phishing is a type of cybercrime where an attacker attempts to trick or deceive a victim into revealing sensitive information, such as passwords, credit card numbers, or personal data. This is typically done through fake emails, messages, or websites that appear to be legitimate, but are actually designed to steal the victim's information.

Phishing attacks often involve:

1. **Spoofing**: Creating a fake email or message that appears to be from a trusted source, such as a bank or a well-known company.
2. **Deception**: Using convincing language and graphics to make the victim believe the message is genuine.
3. **Urgency**: Creating a sense of urgency or panic to prompt the victim into taking action quickly, without thinking twice.

Common types of phishing attacks include:

1. **Email phishing**: Fake emails that ask the victim to click on a link or download an attachment.
2. **Spear phishing**: Targeted attacks on specific individuals or organizations.
3. **Whaling**: Phishing attack

In [49]:
import langchain
import langchain_groq

print("LangChain:", langchain.__version__)
print("LangChain Groq:", langchain_groq.__version__)

LangChain: 1.3.14
LangChain Groq: 1.1.3


In [50]:
from langchain_core.tools import tool

In [51]:
from langchain_core.tools import tool

@tool
def sender_tool(sender: str) -> str:
    """Analyze sender email address."""
    return str(analyze_sender(sender))


@tool
def url_tool(url: str) -> str:
    """Analyze URL."""
    return str(analyze_url(url))


@tool
def content_tool(body: str) -> str:
    """Analyze email content."""
    return str(analyze_content(body))


@tool
def attachment_tool(attachments: str) -> str:
    """Analyze attachments."""
    return str(analyze_attachments(eval(attachments)))

In [52]:
%pip install -U langgraph

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [53]:
from langgraph.prebuilt import create_react_agent

In [54]:
agent = create_react_agent(
    model=llm,
    tools=[
        sender_tool,
        url_tool,
        content_tool,
        attachment_tool,
    ]
)

C:\Users\ARPAN\AppData\Local\Temp\ipykernel_15744\3425282241.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [55]:
print(sender_tool)
print(url_tool)
print(content_tool)
print(attachment_tool)

name='sender_tool' description='Analyze sender email address.' args_schema=<class 'langchain_core.utils.pydantic.sender_tool'> func=<function sender_tool at 0x0000023CDF7EE840>
name='url_tool' description='Analyze URL.' args_schema=<class 'langchain_core.utils.pydantic.url_tool'> func=<function url_tool at 0x0000023CDF7EE7A0>
name='content_tool' description='Analyze email content.' args_schema=<class 'langchain_core.utils.pydantic.content_tool'> func=<function content_tool at 0x0000023CDF7EE8E0>
name='attachment_tool' description='Analyze attachments.' args_schema=<class 'langchain_core.utils.pydantic.attachment_tool'> func=<function attachment_tool at 0x0000023CDF7EEF20>


In [56]:
from langgraph.prebuilt import create_react_agent

print(create_react_agent)

<function create_react_agent at 0x0000023CDFC979C0>


In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
Analyze this phishing email.

Sender:
support@amaz0n-security.xyz

URL:
http://amaz0n-login-security.xyz

Body:
Your account has been suspended.
Verify your password immediately.
Failure to verify within 24 hours will permanently suspend your account.

Attachments:
['invoice.pdf.exe']
"""
            }
        ]
    }
)

In [ ]:
print(response["messages"][-1].content)

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_KEY"))

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
